# 🦷 Session 2 — Implant Brand Recognition: Code → Computer Vision → No-Code

## Dental AI Workshop

This guided Colab lab uses the **same small implant-recognition teaching dataset used in Session 2** with the verified class names:

- **Bego**
- **Bicon**
- **ITI**

You will move through the full computer-vision workflow:

**Dataset → Inspect → Prevent leakage → Split → Train → Evaluate → Predict → Compare with Landing AI**

> **Educational use only.** This notebook is a teaching demonstration, not a clinically validated implant-identification system. Use public, synthetic, or appropriately de-identified data only.

## 🎯 Learning objectives

By the end of this notebook, you should be able to:

1. Explain the difference between **classification, detection, and segmentation**.
2. Inspect an image dataset before training.
3. Recognize why augmented copies of the same source image must not be split across train and test sets.
4. Train a lightweight deep-learning image classifier using transfer learning.
5. Interpret accuracy, precision, recall, F1 score, confidence, and a confusion matrix.
6. Test the frozen model on unseen images.
7. Map the code workflow to the equivalent **Landing AI** no-code workflow.

### Why classification here?

Each teaching image belongs to one implant-brand folder. The task is therefore:

> **Given one implant image, predict its brand: Bego, Bicon, or ITI.**

That is **image classification**.

If the task instead required locating multiple implants with bounding boxes, that would be **object detection**. If the implant had to be outlined pixel-by-pixel, that would be **segmentation**.

## 🟦 RUN THIS — Install the required packages

Colab already includes most scientific Python packages. We install/update only the packages needed for this lab.

In [ ]:
!pip -q install -U ultralytics google-api-python-client

## 🟦 RUN THIS — Imports, reproducibility, and device check

In [ ]:
from pathlib import Path
import os, re, random, shutil, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, ConfusionMatrixDisplay, classification_report
from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU — training will be slower.")

## 🟦 RUN THIS — Download the same Session 2 dataset

The course dataset contains three Google Drive folders, one per verified implant brand.

To make the workshop more reliable, this notebook now uses the **authenticated Google Drive API** instead of anonymous `gdown` folder scraping. Colab will ask you to authorize Google Drive access for this session, then it will download the **Bego, Bicon, and ITI** images directly by file ID.

This avoids the intermittent `gdown` error: **Cannot retrieve the public link of the file**.


In [ ]:
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import io

DATA_ROOT = Path("/content/implant_brand_course_dataset")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

CLASS_FOLDERS = {
    "Bego":  "10fFJRArZE_1odiMS18noxS-MgqNbPwwm",
    "Bicon": "11TpBNjQzrBFEZ1HM230aZUQjgWS5zGlu",
    "ITI":   "1-pvH9R9QSsqjNgqWWtC448KYr4BxE_vY",
}

print("Authorizing Google Drive access for this Colab session...")
auth.authenticate_user()
drive_service = build("drive", "v3")

def list_folder_files(folder_id):
    files = []
    page_token = None
    while True:
        response = drive_service.files().list(
            q=f"'{folder_id}' in parents and trashed = false",
            spaces="drive",
            fields="nextPageToken, files(id, name, mimeType, size)",
            pageSize=1000,
            pageToken=page_token,
            supportsAllDrives=True,
            includeItemsFromAllDrives=True,
        ).execute()
        files.extend(response.get("files", []))
        page_token = response.get("nextPageToken")
        if not page_token:
            break
    return files

def download_drive_file(file_id, destination):
    request = drive_service.files().get_media(fileId=file_id)
    with io.FileIO(destination, "wb") as fh:
        downloader = MediaIoBaseDownload(fh, request, chunksize=1024 * 1024)
        done = False
        while not done:
            _, done = downloader.next_chunk()

for class_name, folder_id in CLASS_FOLDERS.items():
    out_dir = DATA_ROOT / class_name
    out_dir.mkdir(parents=True, exist_ok=True)
    remote_files = [
        f for f in list_folder_files(folder_id)
        if f.get("mimeType", "").startswith("image/")
    ]
    print(f"{class_name}: {len(remote_files)} images found in Drive")

    for idx, item in enumerate(remote_files, start=1):
        destination = out_dir / item["name"]
        expected_size = int(item.get("size", 0) or 0)
        if destination.exists() and (expected_size == 0 or destination.stat().st_size == expected_size):
            continue
        download_drive_file(item["id"], str(destination))
        if idx % 10 == 0 or idx == len(remote_files):
            print(f"  downloaded/checked {idx}/{len(remote_files)}")

    local_images = [p for p in out_dir.iterdir() if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}]
    if len(local_images) != len(remote_files):
        raise RuntimeError(
            f"{class_name}: expected {len(remote_files)} images but found {len(local_images)} locally. "
            "Re-run this cell to resume missing downloads."
        )
    print(f"✓ {class_name}: ready ({len(local_images)} images)")

print("\nDataset root:", DATA_ROOT)


### If the Drive authorization/download is interrupted

1. Make sure you are signed into the Google account you use with Colab.
2. Approve the Drive permission request when Colab asks.
3. Re-run the download cell. It checks existing files and resumes only the missing ones.

If your institution blocks Google Drive API access completely, use the Session 2 resource folder as the manual fallback: download the three folders (**Bego, Bicon, ITI**) and place them under:

`/content/implant_brand_course_dataset/`

Do not rename the classes.


## 🟦 RUN THIS — Build a dataset manifest

We inspect filenames, classes, and the likely **pre-augmentation source image**.

Many filenames follow a pattern such as:

`bego-1_jpg.rf.<augmentation-hash>.jpg`

Images that share the same text before `_jpg.rf.` appear to be augmented variants of the same source image. We will keep those variants together during splitting.

In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

records = []
for class_name in CLASS_FOLDERS:
    class_dir = DATA_ROOT / class_name
    for p in sorted(class_dir.rglob("*")):
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
            source_stem = re.split(r"_jpg\.rf\.", p.name, maxsplit=1, flags=re.IGNORECASE)[0]
            records.append({
                "path": str(p),
                "class": class_name,
                "filename": p.name,
                "source_group": f"{class_name}::{source_stem.lower()}",
            })

df = pd.DataFrame(records)
print("Total images:", len(df))
display(df.head())

summary = (
    df.groupby("class")
      .agg(images=("path", "count"), source_groups=("source_group", "nunique"))
      .reset_index()
)
display(summary)

if len(df) == 0:
    raise RuntimeError("No images were found. Check the download step before continuing.")

## 🟩 DISCUSS — Images are not always independent samples

A dataset may contain many augmented images derived from a smaller number of originals.

If augmented versions of the **same original image** appear in both training and test sets, performance may look better than it really is because the model has effectively seen that source image before.

### Workshop rule

**Split related images together.**

In this notebook we can group by the apparent pre-augmentation filename. That prevents *source-image augmentation leakage*.

> This is **not automatically a patient-level split**. If multiple different source images belong to the same patient, a true patient-level split requires patient identifiers, which are not encoded in these filenames.

## 🟦 RUN THIS — Visualize random examples

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(12, 8))

for row_idx, class_name in enumerate(CLASS_FOLDERS):
    subset = df[df["class"] == class_name]
    sample = subset.sample(n=min(4, len(subset)), random_state=SEED)
    for col_idx, (_, row) in enumerate(sample.iterrows()):
        img = Image.open(row["path"]).convert("RGB")
        axes[row_idx, col_idx].imshow(img, cmap="gray")
        axes[row_idx, col_idx].set_title(class_name)
        axes[row_idx, col_idx].axis("off")

for ax in axes.flat:
    if not ax.has_data():
        ax.axis("off")

plt.suptitle("Session 2 implant-brand teaching dataset", fontsize=14)
plt.tight_layout()
plt.show()

## 🟨 YOUR TURN — Predict before running the split

Before running the next cell, answer:

1. If `Bego-1` has three augmented versions, should one go to train and another to test?
2. Why could that inflate the apparent performance?
3. What would be even stronger than source-image grouping if patient IDs were available?

## 🟦 RUN THIS — Leakage-aware train / validation / test split

We split **source groups within each class**, then assign all augmented variants of a source group to the same partition.

Target teaching proportions are approximately:

- **70% train**
- **15% validation**
- **15% test**

With a small dataset, exact percentages may differ because groups cannot be divided fractionally.

In [ ]:
def assign_group_splits(frame, seed=42, train_frac=0.70, val_frac=0.15):
    rng = random.Random(seed)
    assignments = {}

    for class_name in sorted(frame["class"].unique()):
        groups = sorted(frame.loc[frame["class"] == class_name, "source_group"].unique())
        rng.shuffle(groups)

        n = len(groups)
        if n < 3:
            raise ValueError(f"{class_name} has only {n} independent source groups; need at least 3.")

        n_train = max(1, round(n * train_frac))
        n_val = max(1, round(n * val_frac))
        if n_train + n_val >= n:
            n_train = max(1, n - 2)
            n_val = 1

        train_groups = groups[:n_train]
        val_groups = groups[n_train:n_train+n_val]
        test_groups = groups[n_train+n_val:]

        for g in train_groups:
            assignments[g] = "train"
        for g in val_groups:
            assignments[g] = "val"
        for g in test_groups:
            assignments[g] = "test"

    out = frame.copy()
    out["split"] = out["source_group"].map(assignments)
    return out

split_df = assign_group_splits(df, seed=SEED)

display(pd.crosstab(split_df["class"], split_df["split"]))
print("\nIndependent source groups:")
display(pd.crosstab(split_df["class"], split_df.groupby("source_group")["split"].first().reindex(split_df["source_group"].unique()).values) if False else
        split_df.drop_duplicates("source_group").groupby(["class","split"]).size().unstack(fill_value=0))

# Verify no source group crosses partitions
crossing = split_df.groupby("source_group")["split"].nunique()
assert crossing.max() == 1, "Leakage check failed: a source group appears in multiple splits."
print("✓ Leakage check passed: each source group appears in only one split.")

## 🟦 RUN THIS — Create the folder structure Ultralytics expects

Ultralytics image classification uses:

```
dataset/
  train/
    Bego/
    Bicon/
    ITI/
  val/
    Bego/
    Bicon/
    ITI/
  test/
    Bego/
    Bicon/
    ITI/
```

In [ ]:
SPLIT_ROOT = Path("/content/implant_brand_split")

if SPLIT_ROOT.exists():
    shutil.rmtree(SPLIT_ROOT)

for split_name in ["train", "val", "test"]:
    for class_name in CLASS_FOLDERS:
        (SPLIT_ROOT / split_name / class_name).mkdir(parents=True, exist_ok=True)

for _, row in split_df.iterrows():
    src = Path(row["path"])
    dst = SPLIT_ROOT / row["split"] / row["class"] / src.name
    shutil.copy2(src, dst)

print("Prepared:", SPLIT_ROOT)
for split_name in ["train", "val", "test"]:
    counts = {c: len(list((SPLIT_ROOT/split_name/c).glob("*"))) for c in CLASS_FOLDERS}
    print(split_name, counts)

## 🟦 RUN THIS — Load a pretrained deep-learning model

We use **transfer learning** rather than training a large neural network from zero.

The model starts with visual features learned from a much larger general-image dataset, then adapts to our small implant-brand teaching dataset.

For speed and workshop stability we use:

`yolo11n-cls.pt`

This is the lightweight YOLO11 **classification** model.

In [ ]:
model = YOLO("yolo11n-cls.pt")
model.info()

## 🟨 YOUR TURN — Choose one training setting

You can safely experiment with one variable.

For the first run, keep:

- epochs = 10
- image size = 224
- batch = 16

Later you can change **one** variable and compare.

Do not interpret a higher score from repeated experimentation on the same small test set as proof of better clinical performance.

In [ ]:
EPOCHS = 10       # Try 5, 10, or 20
IMG_SIZE = 224    # Try 224 or 320
BATCH = 16        # Reduce if you run out of memory

print("Epochs:", EPOCHS)
print("Image size:", IMG_SIZE)
print("Batch:", BATCH)
print("Device:", DEVICE)

## 🟦 RUN THIS — Train the classifier

In [ ]:
train_results = model.train(
    data=str(SPLIT_ROOT),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    seed=SEED,
    device=DEVICE,
    workers=2,
    patience=5,
    project="/content/runs",
    name="implant_brand_session2",
    exist_ok=True,
    verbose=True,
)

## 🟩 DISCUSS — What just happened?

The model did not "memorize dentistry" in a human sense.

It adjusted its parameters to separate visual patterns associated with the three training labels.

Ask:

- Are the classes balanced?
- Are the source images representative?
- Could acquisition device, cropping, contrast, or preprocessing become shortcuts?
- Are test images truly independent?
- Would this generalize to another hospital, radiography system, or patient population?

**Good internal performance is not the same as clinical validation.**

## 🟦 RUN THIS — Load the best frozen model

Training normally saves the best checkpoint based on validation performance.

We now load that checkpoint and treat it as the **frozen model** before evaluating the test set.

In [ ]:
BEST_WEIGHTS = Path("/content/runs/implant_brand_session2/weights/best.pt")

if not BEST_WEIGHTS.exists():
    raise FileNotFoundError(f"Best model not found at {BEST_WEIGHTS}")

best_model = YOLO(str(BEST_WEIGHTS))
print("✓ Frozen model loaded:", BEST_WEIGHTS)

## 🟦 RUN THIS — Evaluate on the unseen test partition

We calculate:

- Accuracy
- Precision
- Recall
- F1 score
- Confusion matrix

Macro averaging gives each implant class equal importance regardless of its image count.

In [ ]:
test_rows = split_df[split_df["split"] == "test"].copy()

y_true = []
y_pred = []
confidences = []

for _, row in test_rows.iterrows():
    result = best_model.predict(row["path"], imgsz=IMG_SIZE, device=DEVICE, verbose=False)[0]
    pred_idx = int(result.probs.top1)
    pred_name = result.names[pred_idx]
    confidence = float(result.probs.top1conf)

    y_true.append(row["class"])
    y_pred.append(pred_name)
    confidences.append(confidence)

test_rows["prediction"] = y_pred
test_rows["confidence"] = confidences
test_rows["correct"] = test_rows["class"] == test_rows["prediction"]

labels = sorted(CLASS_FOLDERS.keys())
acc = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, labels=labels, average="macro", zero_division=0
)

print(f"Accuracy:        {acc:.3f}")
print(f"Macro precision: {precision:.3f}")
print(f"Macro recall:    {recall:.3f}")
print(f"Macro F1:        {f1:.3f}")
print("\nClassification report:")
print(classification_report(y_true, y_pred, labels=labels, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(cmap="Blues", values_format="d")
plt.title("Implant brand — test confusion matrix")
plt.show()

## 🟩 DISCUSS — Which metric matters?

- **Accuracy**: overall proportion correct.
- **Precision**: when the model predicts a brand, how often is that prediction correct?
- **Recall**: of all images truly belonging to a brand, how many did the model identify?
- **F1**: balances precision and recall.
- **Confusion matrix**: shows which brands are being confused with which.

There is no universal "best" metric. Interpretation depends on the intended use and consequences of errors.

For this workshop, the goal is to understand evaluation — **not to establish clinical validity from 150 teaching images**.

## 🟦 RUN THIS — Review low-confidence and incorrect cases

Model confidence can help identify cases worth human review, but a high confidence score does **not** prove correctness.

In [ ]:
review = test_rows.sort_values(["correct", "confidence"], ascending=[True, True]).reset_index(drop=True)
display(review[["filename", "class", "prediction", "confidence", "correct"]].head(20))

to_show = review.head(min(12, len(review)))
cols = 4
rows = int(np.ceil(len(to_show) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(14, 3.5 * rows))
axes = np.array(axes).reshape(-1)

for ax, (_, row) in zip(axes, to_show.iterrows()):
    img = Image.open(row["path"]).convert("RGB")
    ax.imshow(img)
    ax.set_title(
        f"True: {row['class']}\nPred: {row['prediction']} ({row['confidence']:.2f})",
        fontsize=9
    )
    ax.axis("off")

for ax in axes[len(to_show):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

## 🟨 YOUR TURN — Human vs AI

Before revealing the model output for a selected test image:

1. Look at the image.
2. Make your own guess: **Bego, Bicon, or ITI**.
3. Run the next cell.
4. Compare your answer with the model and the reference folder label.
5. Discuss disagreement rather than assuming either human or AI is automatically correct.

In [ ]:
CASE_INDEX = 0  # Change this number to test another unseen image

case = test_rows.reset_index(drop=True).iloc[CASE_INDEX]
img = Image.open(case["path"]).convert("RGB")

plt.figure(figsize=(5, 5))
plt.imshow(img)
plt.axis("off")
plt.title("Unseen test image — make your guess before revealing")
plt.show()

input("Press Enter to reveal the model prediction... ")

result = best_model.predict(case["path"], imgsz=IMG_SIZE, device=DEVICE, verbose=False)[0]
pred_idx = int(result.probs.top1)
pred_name = result.names[pred_idx]
confidence = float(result.probs.top1conf)

print("Reference label:", case["class"])
print("Model prediction:", pred_name)
print(f"Model confidence: {confidence:.3f}")

## 🟦 RUN THIS — Export the test results

This CSV can be used for workshop discussion or later statistical analysis.

In [ ]:
RESULTS_CSV = "/content/session2_implant_test_predictions.csv"
test_rows[["filename","source_group","class","prediction","confidence","correct"]].to_csv(RESULTS_CSV, index=False)
print("Saved:", RESULTS_CSV)

from google.colab import files
# Uncomment the next line if you want to download the results:
# files.download(RESULTS_CSV)

## 🔁 Code → No-Code mapping

Now reproduce the same reasoning in **Landing AI**.

| Code notebook | No-code equivalent |
|---|---|
| Collect Bego / Bicon / ITI folders | Upload images |
| Verify labels | Create/check classes |
| Inspect examples | Browse dataset |
| Leakage-aware splitting | Check dataset split strategy |
| Load pretrained model | Start model training |
| Train | Train |
| Accuracy / precision / recall / F1 | Review evaluation dashboard |
| Confusion matrix | Review class confusions |
| Predict unseen image | Test model on unseen image |
| Review errors | Inspect false/uncertain predictions |

### Key lesson

**No-code does not remove the need to understand data quality, splitting, evaluation, or validation.**

## 🧪 Optional experiment — Change ONE variable

After completing the main run, choose only one experiment:

- Increase `EPOCHS` from 10 → 20
- Increase `IMG_SIZE` from 224 → 320
- Compare with another lightweight classifier

Record:

1. What changed?
2. Did validation performance change?
3. Did test performance change?
4. Did training time change?
5. Is the difference large enough to matter, or could it be random variation from a small dataset?

Avoid repeatedly tuning against the test set. The more decisions you make after looking at test performance, the less "unseen" that test set becomes.

## ⚠️ Limitations and responsible interpretation

This workshop dataset is intentionally small.

Important limitations:

- Only three implant-brand labels are used here: **Bego, Bicon, ITI**.
- The dataset contains augmented variants; this notebook groups them by apparent source-image filename to reduce augmentation leakage.
- The filenames do not provide reliable patient IDs, so this is not guaranteed to be a patient-level split.
- A small internal test partition gives an unstable performance estimate.
- Images may not represent other institutions, scanners, acquisition protocols, populations, implant systems, or clinical conditions.
- Confidence is not certainty.
- Clinical deployment would require substantially stronger data, external validation, expert evaluation, bias/generalizability assessment, safety analysis, and appropriate governance.

### Workshop conclusion

**Image → Label → Split correctly → Train → Evaluate → Test unseen data → Inspect errors → Validate externally**

The goal is not simply to obtain a high score. The goal is to understand **why the score can or cannot be trusted**.

## ✅ Session 2 checkpoint

You should now be able to answer:

- Is implant-brand recognition here classification, detection, or segmentation?
- Why must augmented copies of the same source image stay in one split?
- Why is a patient-level split stronger when patient IDs are available?
- What does a confusion matrix reveal?
- Why is high confidence not proof of correctness?
- Why does a small internal test set not establish clinical validity?
- How does this code workflow map to Landing AI?

**Next:** use the same computer-vision reasoning in the Session 2 no-code lab.